In [1]:
from IPython.display import display, Markdown

display(Markdown(r'''
$$
\begin{aligned}

\textbf{Return on carry trade:} \quad 
& r_{S_T} \cdot \frac{Y_T}{Y_0} - e^{r_{B_0}T} \\[10pt]

\textbf{Product payoff structure:} \quad 
& \Phi(r_{S_T}, Y_T) =
\begin{cases}
0, 
& \text{if } X > 0 \\

k(-0.5)X, 
& \text{if } -5\% < X < 0 \\

-0.5X, 
& \text{if } X < -5\%
\end{cases}
\\[6pt]
& \text{where } X = r_{S_T} \frac{Y_T}{Y_0} - e^{r_{B_0}T}, \quad
k = \frac{X}{-5\%} \text{ (proportion of loss compensated)}
\\[12pt]

\textbf{Interest rate processes:} \quad 
& \begin{cases}
\text{US: } dr_{A_t} = \kappa_A (\theta_A - r_{A_t})\,dt + \sigma_A dW_t^A \\
\text{Japan: } dr_{B_t} = \kappa_B (\theta_B - r_{B_t})\,dt + \sigma_B dW_t^B
\end{cases} \\[12pt]
                 
\textbf{Foreign exchange rate SDE:} \quad 
& dY_t = (r_{B_t} - r_{A_t}) Y_t\,dt + \sigma_Y Y_t\,dW_t^Y \\[10pt]

\textbf{Stock price process:} \quad 
& dS_t = r_{A_t} S_t\,dt + \sigma_S S_t\,dW_t^S

\end{aligned}
$$

$$
A = \text{USD}, \quad B = \text{JPY}, \quad Y_0 = \text{USD/JPY at } t=0, \quad Y_T = \text{USD/JPY at } t=T
$$
'''))


$$
\begin{aligned}

\textbf{Return on carry trade:} \quad 
& r_{S_T} \cdot \frac{Y_T}{Y_0} - e^{r_{B_0}T} \\[10pt]

\textbf{Product payoff structure:} \quad 
& \Phi(r_{S_T}, Y_T) =
\begin{cases}
0, 
& \text{if } X > 0 \\

k(-0.5)X, 
& \text{if } -5\% < X < 0 \\

-0.5X, 
& \text{if } X < -5\%
\end{cases}
\\[6pt]
& \text{where } X = r_{S_T} \frac{Y_T}{Y_0} - e^{r_{B_0}T}, \quad
k = \frac{X}{-5\%} \text{ (proportion of loss compensated)}
\\[12pt]

\textbf{Interest rate processes:} \quad 
& \begin{cases}
\text{US: } dr_{A_t} = \kappa_A (\theta_A - r_{A_t})\,dt + \sigma_A dW_t^A \\
\text{Japan: } dr_{B_t} = \kappa_B (\theta_B - r_{B_t})\,dt + \sigma_B dW_t^B
\end{cases} \\[12pt]

\textbf{Foreign exchange rate SDE:} \quad 
& dY_t = (r_{B_t} - r_{A_t}) Y_t\,dt + \sigma_Y Y_t\,dW_t^Y \\[10pt]

\textbf{Stock price process:} \quad 
& dS_t = r_{A_t} S_t\,dt + \sigma_S S_t\,dW_t^S

\end{aligned}
$$

$$
A = \text{USD}, \quad B = \text{JPY}, \quad Y_0 = \text{USD/JPY at } t=0, \quad Y_T = \text{USD/JPY at } t=T
$$


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

START = "2016-03-31"
END = "2026-03-31"

# 读取 NASDQAQ 指数数据
nasdaq = yf.download("^IXIC", start=START, end=END)
nasdaq.columns = nasdaq.columns.droplevel(1)

# 读取 USD/JPY（注意 ticker 是 JPY=X）
usd_jpy = yf.download("JPY=X", start=START, end=END)
usd_jpy.columns = usd_jpy.columns.droplevel(1)

display(nasdaq)  
display(usd_jpy) 

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2016-03-31,4869.850098,4891.299805,4864.410156,4869.569824,1774270000
2016-04-01,4914.540039,4917.089844,4832.060059,4842.549805,1812250000
2016-04-04,4891.799805,4917.750000,4885.169922,4911.209961,1705730000
2016-04-05,4843.930176,4872.700195,4838.620117,4855.899902,1726010000
2016-04-06,4920.720215,4921.509766,4849.279785,4849.580078,1761770000
...,...,...,...,...,...
2026-03-24,21761.890625,21916.160156,21712.039062,21807.599609,8488380000
2026-03-25,21929.820312,22093.179688,21865.460938,22006.429688,8056920000
2026-03-26,21408.080078,21823.580078,21395.769531,21693.179688,7726590000


Price,Close,High,Low,Open,Volume
Date,,,,,
2016-03-31,112.365997,112.639000,112.110001,112.345001,0
2016-04-01,112.480003,112.480003,111.862999,112.480003,0
2016-04-04,111.722000,111.733002,111.135002,111.705002,0
2016-04-05,111.250000,111.244003,110.042999,111.244003,0
2016-04-06,110.377998,110.628998,109.634003,110.369003,0
...,...,...,...,...,...
2026-03-24,158.479004,158.936996,158.408997,158.490005,0
2026-03-25,158.718002,159.225006,158.576996,158.710007,0
2026-03-26,159.384003,159.688995,159.287003,159.401001,0


In [3]:
# Calculate daily volatility for NASDAQ and USD/JPY using log returns
price = nasdaq["Close"].values
log_price = np.log(price)  # 此行没有经济意义
log_return = np.diff(log_price)  # size比原数据小1
sigma_S = np.std(log_return, ddof=1)   # 默认ddof=0计算总体std，这里用ddof=1计算样本std，除以n-1
print("Estimated sigma_S (NASDAQ):", sigma_S)

# 浓缩写法
sigma_Y = np.std(np.diff(np.log(usd_jpy["Close"])), ddof=1)
print("Estimated sigma_Y (USD/JPY):", sigma_Y)

Estimated sigma_S (NASDAQ): 0.013847288840989729
Estimated sigma_Y (USD/JPY): 0.005675218564377177


In [4]:
us_bond = pd.read_csv("US 3-Month Bond Yield.csv")
us_bond = us_bond.rename(columns={"observation_date": "Date", "DTB3": "us"})
us_bond["Date"] = pd.to_datetime(us_bond["Date"])
us_bond["us"] = us_bond["us"]/100
us_bond = us_bond.set_index("Date")

jpn_bond = pd.read_csv("Japan 3-Month Bond Yield.csv", usecols=["Date", "Price"])
jpn_bond["Date"] = pd.to_datetime(jpn_bond["Date"])
jpn_bond["Price"] = jpn_bond["Price"]/100
jpn_bond = jpn_bond.rename(columns={"Price": "jp"})
jpn_bond = jpn_bond.set_index("Date")
jpn_bond = jpn_bond.sort_index()
display(us_bond, jpn_bond)

yields = pd.concat([us_bond, jpn_bond], axis=1, join="inner").dropna()
display(yields)

,us
Date,
2016-03-31,0.0021
2016-04-01,0.0023
2016-04-04,0.0023
2016-04-05,0.0023
2016-04-06,0.0023
...,...
2026-03-25,0.0363
2026-03-26,0.0363
2026-03-27,0.0362


,jp
Date,
2016-03-31,-0.00071
2016-04-01,-0.00071
2016-04-04,-0.00071
2016-04-05,-0.00071
2016-04-06,-0.00075
...,...
2026-03-25,0.00816
2026-03-26,0.00818
2026-03-27,0.00833


,us,jp
Date,,
2016-03-31,0.0021,-0.00071
2016-04-01,0.0023,-0.00071
2016-04-04,0.0023,-0.00071
2016-04-05,0.0023,-0.00071
2016-04-06,0.0023,-0.00075
...,...,...
2026-03-25,0.0363,0.00816
2026-03-26,0.0363,0.00818
2026-03-27,0.0362,0.00833


In [ ]:
# Estimate parameters for each yield series
dt = 1/252
def get_parameters(col):
    r_t = yields[col].values
    r_prev = r_t[:-1]   # r_{t-1}
    r_curr = r_t[1:]    # r_{t}
    d_r_t = r_curr - r_prev
    
    # r_t ​− r_{t−1}​ = κ (r_{t−1} - θ ​) * dt + σ * sqrt(dt) * ε_t​
    #      Δr_t     = κ*r_{t−1}*dt - κ*θ*dt + error
    # 回归: Δr_t     =  b * r_{t-1} + a      + error
    reg = LinearRegression().fit(r_prev.reshape(-1, 1), d_r_t)

    a = reg.intercept_
    b = reg.coef_[0]

    kappa = b / dt
    theta = -a / b

    residuals = d_r_t - reg.predict(r_prev.reshape(-1, 1)) 
    sigma = np.std(residuals, ddof=1) / np.sqrt(dt)

    # Print the estimated parameters
    print(f"For {col}, estimated parameters are:")
    print("κ (kappa):", kappa)
    print("θ (theta):", theta)
    print("σ (sigma):", sigma)

get_parameters("us")
get_parameters("jp")

For us, estimated parameters are:
κ (kappa): -0.0700903059693999
θ (theta): 0.07303214899396632
σ (sigma): 0.00496616586636715
For jp, estimated parameters are:
κ (kappa): -1.7201871884675866
θ (theta): -8.569540223134506e-05
σ (sigma): 0.0052880773038487


读取NASDAQ的option price以获取ATM options的implied volatility

导出数据

import json
params = {"sigma_S": sigma_S, "sigma_Y": sigma_Y}
with open("params.json", "w") as f:
    json.dump(params, f)